In [ ]:
import tqdm
import pickle
import warnings
import numpy as np
import pandas as pd
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from data import *
from model import *
from utils import *
from recourse_model import ROAR, LAROAR, RecourseCost, RobustRecourse

warnings.filterwarnings('ignore')

In [ ]:
def append_res(d, rob, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['Cost'].append(cost)
    d['M1 Validity'].append(m1_validity)
    d['WC Validity'].append(wc_validity)
    d['M1 Expectation'].append(m1_expectation)
    d['WC Expectation'].append(wc_expectation)
    d['J'].append(rob) 
    d['Loss'].append(loss)
    
def get_res(d, alg, seed, alpha, lamb):
    result = {
        'alg': alg, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb}
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [ ]:
def recourse_model_runner(X: np.ndarray, recourse_model1: LAROAR, recourse_model2: ROAR, base_model: Model, m1, X_train, seed):    
    alpha = recourse_model1.alpha
    lamb = recourse_model1.lamb
    
    model_adv_opt = deepcopy(base_model)
    model_adv_roar = deepcopy(base_model)
    
    results_opt = {'Cost': [], 'M1 Validity': [], 'WC Validity': [], 'M1 Expectation': [], 'WC Expectation': [], 'J': [], 'Loss': []}
    results_roar = deepcopy(results_opt)
    
    n = len(X)

    for i in tqdm.trange(n, desc=f'Eval alpha={alpha}; lambda={lamb}', colour='#0091ff'):
    # for i in range(n):
        x_0 = X[i]
        J = RecourseCost(x_0, lamb)
        
        if isinstance(base_model, NN):
            #set seed for lime
            np.random.seed(i)
            weights_0, bias_0 = lime_explanation(base_model.predict, X_train, x_0)
            weights_0, bias_0 = np.round(weights_0, 4), np.round(bias_0, 4)
            
            recourse_model1.weights = weights_0
            recourse_model1.bias = bias_0
            
            recourse_model2.set_weights(weights_0)
            recourse_model2.set_bias(bias_0)
        
        # OPT
        x_r = recourse_model1.get_recourse(x_0, beta=1.)
        weights_r, bias_r = recourse_model1.calc_theta_adv(x_r)
        bce_loss_opt, cost_opt, rob_opt = J(x_r, weights_r, bias_r, True)
        m1_validity_opt = base_model.predict(x_r.reshape(1,-1))[0]
        m1_expectation_opt = base_model.predict_proba(x_r.reshape(1,-1))[0,1]
        
        model_adv_opt.model.coef_ = weights_r.reshape(1,-1)
        model_adv_opt.model.intercept_ = bias_r
        wc_validity_opt = model_adv_opt.predict(x_r.reshape(1,-1))[0]
        wc_expectation_opt = model_adv_opt.predict_proba(x_r.reshape(1,-1))[0,1]
        
        # ROAR
        x_r2, _ = recourse_model2.get_recourse(x_0)
        weights_r2, bias_r2 = recourse_model1.calc_theta_adv(x_r2)
        bce_loss_roar, cost_roar, rob_roar = J(x_r2, weights_r2, bias_r2, True)
        m1_validity_roar = base_model.predict(x_r2.reshape(1,-1))[0]
        m1_expectation_roar = base_model.predict_proba(x_r2.reshape(1,-1))[0,1]
        
        model_adv_roar.model.coef_ = weights_r2.reshape(1,-1)
        model_adv_roar.model.intercept_ = bias_r2
        wc_validity_roar = model_adv_roar.predict(x_r2.reshape(1,-1))[0]
        wc_expectation_roar = model_adv_roar.predict_proba(x_r2.reshape(1,-1))[0,1]
        
        append_res(results_opt, rob_opt, bce_loss_opt, cost_opt, m1_validity_opt, wc_validity_opt, m1_expectation_opt, wc_expectation_opt)
        append_res(results_roar, rob_roar, bce_loss_roar, cost_roar, m1_validity_roar, wc_validity_roar, m1_expectation_roar, wc_expectation_roar)

    return get_res(results_opt, 'OPT', seed, alpha, lamb), get_res(results_roar, 'ROAR', seed, alpha, lamb)

In [ ]:
def recourse_tradeoff(params: dict, seeds: list, results: list):
    if params['data'] in ['correction', 'german']:
        data_model = CorrectionShift("../datasets/german.csv", "../datasets/corrected_german.csv", seed=0)
    elif params['data'] in ['temporal', 'business']:
        data_model = TemporalShift("../datasets/SBAcase.11.13.17.csv", seed=0)
    elif params['data'] in ['geospatial', 'student']:
        data_model = GeospatialShift("../datasets/student-por.csv", seed=0)
    elif params['data'] in ['synthetic', 'simulated']:
        data_model = SyntheticData(alpha=1.5, beta=0, n=1000, seed=0)
    
    alpha = params['alpha']
    
    for seed in seeds:
        data1, _ = data_model.get_data(seed)
        X1_train, y1_train, X1_test, y1_test = data1

        if params['base_model'] == 'lr':
            base_model = LR()
            m1 = deepcopy(base_model)
        elif params['base_model'] == 'nn':
            base_model = NN(X1_train.shape[1])
            m1 = LR()

        base_model.train(X1_train.values, y1_train.values)
        m1.train(X1_train.values, y1_train.values)
        
        recourse_needed_X1_train = recourse_needed(base_model.predict, X1_train.values)
        recourse_needed_X1_test = recourse_needed(base_model.predict, X1_test.values)
        
        print(len(recourse_needed_X1_train), len(X1_train), len(recourse_needed_X1_test), len(X1_test))
        
        weights, bias = None, None
        if params['base_model'] == 'lr':
            weights = base_model.model.coef_[0]
            bias = base_model.model.intercept_
        
        recourse_model1 = LAROAR(
            weights = weights,
            bias = bias,
            alpha = alpha,
        )    
        
        recourse_model2 = ROAR(
            weights = weights,
            bias = bias,
            alpha = alpha,
        )
        
        lamb = recourse_model1.choose_lambda(recourse_needed_X1_train, base_model.predict, X1_train, base_model.predict_proba)
        recourse_model1.lamb = lamb
        recourse_model2.lamb = lamb
        
        result_opt, result_roar = recourse_model_runner(recourse_needed_X1_test, recourse_model1, recourse_model2, base_model, m1, X1_train, seed)
        
        results.append(result_opt)
        results.append(result_roar)
        
        print()

In [ ]:
torch.manual_seed(0)
params = {}
# 'synthetic/simulated', 'correction/german', 'temporal/business', 'geospatial/student'
params['data'] = 'synthetic'
# 'lr', 'nn
params['base_model'] = 'lr'
 
params['alpha'] = 0.5


seeds = list(range(5))

results = []
recourse_tradeoff(params, seeds, results)
df_results = pd.DataFrame(results)

In [ ]:
df_results = pd.DataFrame(results)
df_results.sort_values('alg')

In [ ]:
df_results_avg = df_results.groupby('alg').mean()
df_results_avg[['Cost', 'M1 Validity', 'WC Validity', 'M1 Expectation', 'WC Expectation',  'J', 'Loss']] = df_results_avg[['Cost', 'M1 Validity', 'WC Validity', 'M1 Expectation', 'WC Expectation', 'J', 'Loss']].round(2).astype(str) + '±' + df_results.groupby('alg').std()[['Cost', 'M1 Validity', 'WC Validity', 'M1 Expectation', 'WC Expectation', 'J', 'Loss']].round(2).astype(str)

print(params['base_model'], params['data'])
df_results_avg

In [ ]:
with open(f'../results_comparison/comparison_{params["base_model"]}_{params["data"]}.pkl', 'wb') as f:
    pickle.dump(df_results, f)